# L2-ARCTIC MDD (Kaggle GPU)

1. Add Data: `tnguynthnh142/l2-arctic1` (`ABA/ABA/wav/` layout)
2. GPU + Internet ON → Run All → download `l2_mdd_best.pt`
_OOM: `batch_size=2`, `grad_accum_steps=8`_

In [ ]:
!pip uninstall -y -q torchao
!pip install -q transformers torchaudio torch-geometric peft tqdm textgrid

In [ ]:
# Bypass xung dot torchao 0.10.0 cu tren Kaggle voi PEFT moi
try:
    import peft.import_utils
    peft.import_utils.is_torchao_available = lambda: False
except Exception:
    pass

CONFIG = {
  "wavlm": {
    "model_name": "microsoft/wavlm-large",
    "freeze": True,
    "use_lora": True,
    "lora_r": 8,
    "lora_alpha": 16,
    "lora_dropout": 0.05,
    "lora_target_modules": ["intermediate_dense", "output_dense"],
  },
  "transformer": {"num_layers": 3, "num_heads": 8, "ff_dim": 2048, "dropout": 0.2, "max_seq_len": 800},
  "ctc_align": {"use_wavlm_ctc_head": True},
  "phoneme_graph": {
    "hidden_dim": 256,
    "num_gat_layers": 2,
    "num_heads": 4,
    "dropout": 0.2,
    "edge_types": {"sequential": True, "same_word": True, "same_syllable": False},
  },
  "mdd": {
    "num_classes": 4,
    "dropout": 0.2,
    "canon_emb_dim": 256,
    "class_weights": None,  # Auto-computed from train distribution: (N_0 / N_c)^0.55 clamped [1.0, 8.0]
    "focal_gamma": 1.5,
  },
  "paths": {"checkpoint_dir": "/kaggle/working/checkpoints/l2_mdd"},
  "train": {
    "batch_size": 4,
    "grad_accum_steps": 4,
    "num_epochs": 25,
    "learning_rate": 1.5e-4,
    "learning_rate_wavlm": 4.0e-5,
    "weight_decay": 0.02,
    "warmup_epochs": 2,
    "early_stopping_patience": 6,
    "grad_clip": 1.0,
    "num_workers": 2,
    "pin_memory": True,
    "use_amp": True,
    "ctc_weight": 0.5,
    "save_every_epochs": 5,
    "dataset": {"data_dir": None, "max_duration_sec": 10.0, "sample_rate": 16000},
  },
}

import re
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import torch
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import Dataset

KAGGLE_ROOT = Path("/kaggle/input/datasets/tnguynthnh142/l2-arctic1")
PROBE = ("ABA", "TLV", "NJS")

TEST_SPEAKERS = frozenset({"TLV", "NJS", "TNI", "TXHC", "ZHAA", "YKWK"})
ALL_SPEAKERS = [
    "ABA", "SKA", "YBAA", "ZHAA", "BWC", "LXC", "NCC", "TXHC", "ASI", "RRBI",
    "SVBI", "TNI", "HJK", "HKK", "YDCK", "YKWK", "EBVS", "ERMS", "MBMPS", "NJS",
    "HQTV", "PNV", "THV", "TLV",
]

ERR_CORRECT, ERR_SUB, ERR_DEL, ERR_ADD = 0, 1, 2, 3
ERR_NAMES = ("correct", "substitution", "deletion", "addition")

DEFAULT_DATA_DIR = None
SAMPLE_RATE = 16000

_SIL_TOKENS = frozenset({"", "sil", "sp", "spn", "pau", "nsn"})
_ARPABET_VOWELS = frozenset({
    "AA", "AE", "AH", "AO", "AW", "AY", "EH", "ER", "EY", "IH", "IY", "OW", "OY", "UH", "UW"
})
_PHONEME_ALIASES: Dict[str, str] = {
    "AX": "AH0", "AX0": "AH0", "AX1": "AH0", "AX2": "AH0",
    "AXR": "ER0", "AXR0": "ER0", "AXR1": "ER0", "AXR2": "ER0",
    "IX": "IH0", "IX0": "IH0", "IX1": "IH0", "IX2": "IH0",
    "UX": "UW0", "UX0": "UW0", "UX1": "UW0", "UX2": "UW0",
    "DX": "T", "NX": "N", "EL": "L", "EM": "M", "EN": "N",
    "ENG": "NG", "HV": "HH", "WH": "W", "Q": "T",
}


def normalize_arpabet(ph: Optional[str]) -> Optional[str]:
    """Normalize L2-ARCTIC TextGrid phone string to CMU/SpeechOcean uppercase ARPAbet."""
    if not ph:
        return None
    raw = ph.strip()
    if raw.lower() in _SIL_TOKENS:
        return None
    clean = re.sub(r"[^A-Za-z0-9]", "", raw).upper()
    if not clean or clean.lower() in _SIL_TOKENS:
        return None
    if clean in _PHONEME_ALIASES:
        return _PHONEME_ALIASES[clean]
    base = clean.rstrip("0123456789")
    if not base or base.lower() in _SIL_TOKENS:
        return None
    if base in _PHONEME_ALIASES:
        return _PHONEME_ALIASES[base]
    if base in _ARPABET_VOWELS:
        if len(clean) > len(base) and clean[-1] in ("0", "1", "2"):
            return f"{base}{clean[-1]}"
        return f"{base}1"
    return base


def speaker_dir(root: Path, spk: str) -> Path:
    r = Path(root)
    nested = r / spk / spk
    if (nested / "annotation").is_dir() or (nested / "wav").is_dir():
        return nested
    return r / spk


def _ok(root: Path, spk: str) -> bool:
    d = speaker_dir(root, spk)
    return (d / "annotation").is_dir() and (d / "wav").is_dir() and any((d / "annotation").glob("*.TextGrid"))


def prepare_l2_arctic_dir(input_root=None):
    seen, cands = set(), [KAGGLE_ROOT, Path("/kaggle/input/l2-arctic1")]
    base = Path(input_root or "/kaggle/input")
    if base.is_dir():
        for tg in base.rglob("annotation/*.TextGrid"):
            inner = tg.parent.parent
            root = inner.parent.parent if inner.parent.name == inner.name else inner.parent
            cands.append(root)
    for c in cands:
        c = Path(c).resolve()
        if str(c) in seen:
            continue
        seen.add(str(c))
        if any(_ok(c, s) for s in PROBE):
            return c
    raise FileNotFoundError("Add Data: tnguynthnh142/l2-arctic1 (expect .../ABA/ABA/wav/)")


def verify_l2_arctic(root):
    root = Path(root)
    if not any(_ok(root, s) for s in PROBE):
        return False, ["Need .../SPEAKER/SPEAKER/wav/ + annotation/"], {"root": str(root)}
    aba = speaker_dir(root, "ABA")
    return True, [], {"root": str(root), "aba": str(aba), "ann": sum(1 for _ in root.rglob("annotation/*.TextGrid"))}


def _is_sil(ph: Optional[str]) -> bool:
    if ph is None:
        return True
    cleaned = re.sub(r"[^a-zA-Z0-9]", "", ph).lower()
    return cleaned in _SIL_TOKENS


def _parse_mark(mark: str) -> Tuple[Optional[str], Optional[str], str]:
    """Parse L2-ARCTIC phones tier mark -> (canonical_arpabet, perceived_arpabet, error_tag)."""
    if not mark or not mark.strip():
        return None, None, ""
    parts = [p.strip() for p in mark.split(",")]
    if len(parts) == 1:
        if _is_sil(parts[0]):
            return "sil", "sil", ""
        ph = normalize_arpabet(parts[0])
        return ph, ph, ""

    cano_raw = parts[0] if len(parts) > 0 else ""
    perc_raw = parts[1] if len(parts) > 1 else ""
    tag_raw = re.sub(r"[^a-zA-Z]", "", parts[2]).lower() if len(parts) > 2 else ""

    cano = "sil" if _is_sil(cano_raw) else normalize_arpabet(cano_raw)
    perc = "sil" if _is_sil(perc_raw) else normalize_arpabet(perc_raw)
    if tag_raw in ("s", "d", "a"):
        tag = tag_raw
    elif tag_raw.startswith("sub"):
        tag = "s"
    elif tag_raw.startswith("del"):
        tag = "d"
    elif tag_raw.startswith("add") or tag_raw.startswith("ins"):
        tag = "a"
    else:
        tag = ""
    return cano, perc, tag


def _load_tiers(tg_path: str):
    """Return (phones_tier, words_tier_or_None)."""
    try:
        from textgrid import TextGrid

        tg = TextGrid(strict=False)

        def _lenient_append(tier):
            if tg.maxTime is not None and tier.maxTime is not None and tier.maxTime > tg.maxTime:
                tg.maxTime = tier.maxTime
            tier.strict = False
            for interval in tier:
                interval.strict = False
            tg.tiers.append(tier)

        tg.append = _lenient_append
        tg.read(tg_path)
        phones_tier = tg.getFirst("phones")
        try:
            words_tier = tg.getFirst("words")
        except Exception:
            words_tier = None
        return phones_tier, words_tier
    except Exception:
        return _phones_tier_fallback(tg_path), None


def _phones_tier_fallback(tg_path: str):
    text = Path(tg_path).read_text(encoding="utf-8", errors="replace")
    marks: List[str] = []
    in_phones = False
    for line in text.splitlines():
        s = line.strip()
        if re.match(r'name\s*=\s*"phones"', s, re.I):
            in_phones = True
            continue
        if in_phones and re.match(r'name\s*=\s*"', s, re.I) and "phones" not in s.lower():
            break
        if in_phones and re.search(r"text\s*=", s, re.I):
            m = re.search(r'=\s*"(.*)"\s*$', s)
            if m:
                marks.append(m.group(1))
    if not marks:
        return None

    class _Interval:
        __slots__ = ("mark", "minTime", "maxTime")

        def __init__(self, mark: str):
            self.mark = mark
            self.minTime = None
            self.maxTime = None

    return [_Interval(m) for m in marks]


def _word_ranges_from_tier(phone_mids: List[Optional[float]], words_tier) -> Optional[List[Tuple[int, int]]]:
    """Map each kept phoneme midpoint to its containing word interval in TextGrid."""
    if not phone_mids or words_tier is None or any(m is None for m in phone_mids):
        return None
    word_ivs = [
        (float(w.minTime), float(w.maxTime))
        for w in words_tier
        if getattr(w, "mark", None) and not _is_sil(w.mark)
    ]
    if not word_ivs:
        return None

    w_ids: List[int] = []
    w_ptr = 0
    for mid in phone_mids:
        while w_ptr + 1 < len(word_ivs) and word_ivs[w_ptr][1] < mid - 1e-4:
            w_ptr += 1
        ws, we = word_ivs[w_ptr]
        if ws - 1e-4 <= mid <= we + 1e-4:
            w_ids.append(w_ptr)
        else:
            # Assign nearest word so every phoneme participates in a valid word group
            best_idx = min(
                range(len(word_ivs)),
                key=lambda k: max(0.0, word_ivs[k][0] - mid, mid - word_ivs[k][1]),
            )
            w_ids.append(best_idx)

    ranges: List[Tuple[int, int]] = []
    start = 0
    for i in range(1, len(w_ids)):
        if w_ids[i] != w_ids[start]:
            ranges.append((start, i))
            start = i
    ranges.append((start, len(w_ids)))
    return ranges


def parse_textgrid(tg_path: str) -> Tuple[List[str], List[int], Optional[List[Tuple[int, int]]]]:
    phones_tier, words_tier = _load_tiers(tg_path)
    if phones_tier is None:
        return [], [], None

    canonical: List[str] = []
    labels: List[int] = []
    mids: List[Optional[float]] = []

    for interval in phones_tier:
        cano, perc, tag = _parse_mark(interval.mark)
        tmin = getattr(interval, "minTime", None)
        tmax = getattr(interval, "maxTime", None)
        mid = (0.5 * (float(tmin) + float(tmax))) if (tmin is not None and tmax is not None) else None

        # Check Addition BEFORE skipping sil canonical intervals! (L2-ARCTIC adds are written as 'sil,P,a')
        if tag == "a" or (_is_sil(cano) and perc is not None and not _is_sil(perc)):
            if perc is not None and not _is_sil(perc):
                canonical.append(perc)
                labels.append(ERR_ADD)
                mids.append(mid)
            continue

        if cano is None or _is_sil(cano):
            continue

        if tag == "d" or perc is None or _is_sil(perc):
            canonical.append(cano)
            labels.append(ERR_DEL)
            mids.append(mid)
        elif tag == "s" or (perc.rstrip("012") != cano.rstrip("012")):
            canonical.append(cano)
            labels.append(ERR_SUB)
            mids.append(mid)
        else:
            canonical.append(cano)
            labels.append(ERR_CORRECT)
            mids.append(mid)

    word_ranges = _word_ranges_from_tier(mids, words_tier)
    return canonical, labels, word_ranges


def _uniform_word_ranges(phones: List[str], text: str) -> List[Tuple[int, int]]:
    words = text.split()
    if not words:
        return [(0, len(phones))]
    n = len(words)
    chunk = max(len(phones) // n, 1)
    ranges = []
    for i in range(n):
        s = i * chunk
        e = len(phones) if i == n - 1 else min((i + 1) * chunk, len(phones))
        ranges.append((s, e))
    return ranges


def load_l2_arctic_annotated(
    split: str = "train",
    data_dir: Optional[str] = None,
) -> List[Dict[str, Any]]:
    root = Path(data_dir)
    test = split == "test"
    speakers = [s for s in ALL_SPEAKERS if (s in TEST_SPEAKERS) == test]

    samples = []
    for spk in speakers:
        spk_root = speaker_dir(root, spk)
        ann_dir = spk_root / "annotation"
        if not ann_dir.is_dir():
            continue
        for tg in sorted(ann_dir.glob("*.TextGrid")):
            stem = tg.stem
            wav = spk_root / "wav" / f"{stem}.wav"
            txt = spk_root / "transcript" / f"{stem}.txt"
            if not wav.is_file():
                continue
            phones, labels, tg_word_ranges = parse_textgrid(str(tg))
            if not phones:
                continue
            text = txt.read_text(encoding="utf-8").strip() if txt.is_file() else ""
            word_ranges = tg_word_ranges if tg_word_ranges else _uniform_word_ranges(phones, text)
            samples.append(
                {
                    "id": f"{spk}_{stem}",
                    "speaker": spk,
                    "audio": str(wav),
                    "text": text,
                    "phoneme_tokens": phones,
                    "phoneme_labels": labels,
                    "word_phone_ranges": word_ranges,
                }
            )
    return samples


class L2ArcticMDDDataset(Dataset):
    def __init__(
        self,
        split: str = "train",
        sample_rate: int = SAMPLE_RATE,
        max_duration_sec: float = 10.0,
        data_dir: Optional[str] = None,
    ):
        self.sample_rate = sample_rate
        self.max_samples = int(max_duration_sec * sample_rate)
        self.samples = load_l2_arctic_annotated(split, data_dir)

    def __len__(self) -> int:
        return len(self.samples)

    def _load_audio(self, path: str) -> torch.Tensor:
        import torchaudio

        wav, sr = torchaudio.load(path)
        wav = wav.mean(0)
        if sr != self.sample_rate:
            wav = torchaudio.functional.resample(wav, sr, self.sample_rate)
        if wav.shape[0] > self.max_samples:
            wav = wav[: self.max_samples]
        return wav

    def __getitem__(self, idx: int) -> Dict[str, Any]:
        item = self.samples[idx]
        return {
            "id": item["id"],
            "waveform": self._load_audio(item["audio"]),
            "text": item["text"],
            "phoneme_tokens": item["phoneme_tokens"],
            "phoneme_labels": item["phoneme_labels"],
            "word_phone_ranges": item["word_phone_ranges"],
        }


def collate_mdd_fn(batch: List[Dict[str, Any]]) -> Dict[str, Any]:
    waveforms = [b["waveform"] for b in batch]
    return {
        "ids": [b["id"] for b in batch],
        "waveforms": pad_sequence(waveforms, batch_first=True),
        "wav_lengths": torch.tensor([w.shape[0] for w in waveforms], dtype=torch.long),
        "texts": [b["text"] for b in batch],
        "phoneme_tokens": [b["phoneme_tokens"] for b in batch],
        "phoneme_labels": [b["phoneme_labels"] for b in batch],
        "word_phone_ranges": [b["word_phone_ranges"] for b in batch],
    }

In [ ]:
import logging
import math
import time
from contextlib import nullcontext
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from tqdm import tqdm
from transformers import WavLMModel

try:
    from torch_geometric.nn import GATv2Conv
except ImportError:
    GATv2Conv = None

torchaudio = None
forced_align = None


def _ensure_torchaudio():
    global torchaudio, forced_align
    if torchaudio is not None:
        return True
    try:
        import torchaudio as _ta
        from torchaudio.functional import forced_align as _fa

        torchaudio = _ta
        forced_align = _fa
        return True
    except (ImportError, OSError):
        return False


class WavLMEncoder(nn.Module):
    def __init__(
        self,
        model_name: str = "microsoft/wavlm-large",
        freeze: bool = True,
        use_lora: bool = False,
        lora_r: int = 8,
        lora_alpha: int = 16,
        lora_dropout: float = 0.05,
        lora_target_modules: Optional[list] = None,
    ):
        super().__init__()
        self.wavlm = WavLMModel.from_pretrained(model_name)
        self.output_dim = self.wavlm.config.hidden_size  # 1024 for large

        if freeze and not use_lora:
            for p in self.wavlm.parameters():
                p.requires_grad = False
            self.wavlm.eval()

        self._frozen = freeze and not use_lora

        if use_lora:
            from peft import LoraConfig, get_peft_model

            target = lora_target_modules or ["q_proj", "v_proj"]
            lora_config = LoraConfig(
                r=lora_r,
                lora_alpha=lora_alpha,
                target_modules=target,
                lora_dropout=lora_dropout,
                bias="none",
            )
            self.wavlm = get_peft_model(self.wavlm, lora_config)

    def forward(
        self,
        waveform: torch.Tensor,
        wav_lengths: Optional[torch.Tensor] = None,
        attention_mask: Optional[torch.Tensor] = None,
    ) -> torch.Tensor:
        if attention_mask is None:
            B, S = waveform.shape
            if wav_lengths is not None:
                attention_mask = (
                    torch.arange(S, device=waveform.device).unsqueeze(0)
                    < wav_lengths.unsqueeze(1)
                ).long()
            else:
                attention_mask = (waveform.abs() > 1e-8).long()

        if self._frozen:
            self.wavlm.eval()

        ctx = torch.no_grad if self._frozen else nullcontext
        with ctx():
            outputs = self.wavlm(
                input_values=waveform,
                attention_mask=attention_mask,
            )
        return outputs.last_hidden_state

    def frame_lengths_from_samples(self, wav_lengths: torch.Tensor) -> torch.Tensor:
        return self.wavlm._get_feat_extract_output_lengths(wav_lengths).long()

    def frame_rate(self, sample_rate: int = 16000) -> float:
        return sample_rate / 320.0


class SinusoidalPositionalEncoding(nn.Module):
    def __init__(self, dim: int, max_len: int = 2000):
        super().__init__()
        pe = torch.zeros(max_len, dim)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(
            torch.arange(0, dim, 2, dtype=torch.float) * (-math.log(10000.0) / dim)
        )
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer("pe", pe.unsqueeze(0), persistent=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x + self.pe[:, : x.size(1), :]


class TaskTransformerEncoder(nn.Module):
    def __init__(
        self,
        input_dim: int,
        num_layers: int = 3,
        num_heads: int = 8,
        ff_dim: int = 4096,
        dropout: float = 0.1,
        max_seq_len: int = 2000,
    ):
        super().__init__()
        if input_dim % num_heads != 0:
            raise ValueError(f"input_dim ({input_dim}) must be divisible by num_heads ({num_heads})")
        self.input_proj = nn.Linear(input_dim, input_dim)
        self.pos_encoding = SinusoidalPositionalEncoding(input_dim, max_seq_len)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=input_dim,
            nhead=num_heads,
            dim_feedforward=ff_dim,
            dropout=dropout,
            batch_first=True,
            norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.output_dim = input_dim

    def forward(
        self,
        x: torch.Tensor,
        src_key_padding_mask: Optional[torch.Tensor] = None,
    ) -> torch.Tensor:
        x = self.input_proj(x)
        x = self.pos_encoding(x)
        return self.encoder(x, src_key_padding_mask=src_key_padding_mask)


DEFAULT_PHONEME_VOCAB = [
    "<pad>", "<unk>", "|",  # blank, unknown, word boundary
    "AA0", "AA1", "AA2", "AE0", "AE1", "AE2", "AH0", "AH1", "AH2",
    "AO0", "AO1", "AO2", "AW0", "AW1", "AW2", "AY0", "AY1", "AY2",
    "B", "CH", "D", "DH", "EH0", "EH1", "EH2", "ER0", "ER1", "ER2",
    "EY0", "EY1", "EY2", "F", "G", "HH", "IH0", "IH1", "IH2",
    "IY0", "IY1", "IY2", "JH", "K", "L", "M", "N", "NG",
    "OW0", "OW1", "OW2", "OY0", "OY1", "OY2", "P", "R", "S", "SH",
    "T", "TH", "UH0", "UH1", "UH2", "UW0", "UW1", "UW2",
    "V", "W", "Y", "Z", "ZH",
]


@dataclass
class PhonemeAlignment:
    phoneme: str
    token_id: int
    start_frame: int
    end_frame: int
    confidence: float


class CTCAligner(nn.Module):
    def __init__(
        self,
        input_dim: int,
        phoneme_vocab: Optional[List[str]] = None,
        use_pretrained_bundle: bool = False,
        blank_id: int = 0,
    ):
        super().__init__()
        self.phoneme_vocab = phoneme_vocab or DEFAULT_PHONEME_VOCAB
        self.token2id = {p: i for i, p in enumerate(self.phoneme_vocab)}
        self.blank_id = blank_id
        self.num_tokens = len(self.phoneme_vocab)
        self.ctc_proj = nn.Linear(input_dim, self.num_tokens)

    def phonemes_to_ids(self, phonemes: List[str]) -> torch.Tensor:
        unk_id = self.token2id["<unk>"]
        ids = []
        for p in phonemes:
            if p in self.token2id:
                ids.append(self.token2id[p])
            else:
                norm = normalize_arpabet(p) if "normalize_arpabet" in globals() else p.upper()
                ids.append(self.token2id.get(norm or "", unk_id))
        return torch.tensor(ids, dtype=torch.long)

    def forward_ctc_logits(self, frame_features: torch.Tensor) -> torch.Tensor:
        logits = self.ctc_proj(frame_features).float()
        log_probs = F.log_softmax(logits, dim=-1)
        return log_probs.transpose(0, 1)

    def _align_log_probs(self, frame_features: torch.Tensor) -> torch.Tensor:
        logits = self.ctc_proj(frame_features.unsqueeze(0)).float()
        return F.log_softmax(logits, dim=-1)

    def _run_forced_align(
        self,
        log_probs: torch.Tensor,
        targets: torch.Tensor,
        input_lengths: torch.Tensor,
        target_lengths: torch.Tensor,
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        assert log_probs.dim() == 3 and log_probs.shape[0] == 1
        try:
            return forced_align(
                log_probs.float(),
                targets.unsqueeze(0),
                input_lengths,
                target_lengths,
                blank=self.blank_id,
            )
        except RuntimeError:
            return forced_align(
                log_probs.float().cpu(),
                targets.unsqueeze(0).cpu(),
                input_lengths.cpu(),
                target_lengths.cpu(),
                blank=self.blank_id,
            )

    def align_utterance(
        self,
        frame_features: torch.Tensor,
        phonemes: List[str],
        frame_lengths: Optional[int] = None,
        log_probs: Optional[torch.Tensor] = None,
    ) -> Tuple[List[PhonemeAlignment], torch.Tensor]:
        if frame_lengths is None:
            frame_lengths = frame_features.shape[0]
        frame_lengths = min(int(frame_lengths), int(frame_features.shape[0]))

        T, D = frame_features.shape
        if len(phonemes) == 0 or frame_lengths == 0:
            return [], frame_features.new_zeros(0, D)

        targets = self.phonemes_to_ids(phonemes).to(frame_features.device)
        t_list = targets.tolist()
        min_ctc_frames = len(t_list) + sum(1 for a, b in zip(t_list[:-1], t_list[1:]) if a == b)

        if not _ensure_torchaudio() or forced_align is None or frame_lengths < min_ctc_frames:
            return self._uniform_align(frame_features[:frame_lengths], phonemes)

        if log_probs is None:
            lp = self._align_log_probs(frame_features[:frame_lengths])
        else:
            lp = log_probs if log_probs.dim() == 3 else log_probs.unsqueeze(0)
            lp = lp[:, :frame_lengths, :].float()

        input_lengths = torch.tensor([frame_lengths], device=lp.device)
        target_lengths = torch.tensor([len(targets)], device=lp.device)

        try:
            aligned_tokens, _ = self._run_forced_align(
                lp, targets.to(lp.device), input_lengths, target_lengths
            )
        except RuntimeError:
            return self._uniform_align(frame_features[:frame_lengths], phonemes)

        aligned = aligned_tokens[0, :frame_lengths] if aligned_tokens.dim() > 1 else aligned_tokens[:frame_lengths]
        spans = self._tokens_to_spans(aligned, targets, phonemes)
        node_features = self._pool_node_features(frame_features[:frame_lengths], spans)
        return spans, node_features

    def _tokens_to_spans(
        self,
        aligned: torch.Tensor,
        targets: torch.Tensor,
        phonemes: List[str],
    ) -> List[PhonemeAlignment]:
        """Decode forced_align token-ID trajectory monotonically into per-phoneme spans."""
        spans: List[PhonemeAlignment] = []
        target_list = targets.tolist()
        N = len(phonemes)
        T = int(aligned.shape[0])
        if N == 0:
            return spans

        arr = aligned.tolist()
        runs: List[Tuple[int, int, int]] = []
        idx = 0
        while idx < T:
            tok = int(arr[idx])
            if tok != self.blank_id:
                s_f = idx
                while idx + 1 < T and int(arr[idx + 1]) == tok:
                    idx += 1
                runs.append((tok, s_f, idx))
            idx += 1

        raw_spans: List[Optional[Tuple[int, int, float]]] = [None] * N
        r_idx = 0
        for i in range(N):
            tid = int(target_list[i])
            if r_idx < len(runs) and runs[r_idx][0] == tid:
                _, s_f, e_f = runs[r_idx]
                raw_spans[i] = (s_f, e_f, min(1.0, (e_f - s_f + 1) / 3.0))
                r_idx += 1
            else:
                found = -1
                for k in range(r_idx, min(len(runs), r_idx + 4)):
                    if runs[k][0] == tid:
                        found = k
                        break
                if found != -1:
                    _, s_f, e_f = runs[found]
                    raw_spans[i] = (s_f, e_f, min(1.0, (e_f - s_f + 1) / 3.0))
                    r_idx = found + 1

        anchors = [(i, raw_spans[i][0], raw_spans[i][1], raw_spans[i][2]) for i in range(N) if raw_spans[i] is not None]
        if not anchors:
            step = T / max(1, N)
            for i in range(N):
                s_f = int(i * step)
                e_f = min(T - 1, int((i + 1) * step))
                spans.append(PhonemeAlignment(phonemes[i], target_list[i], s_f, e_f, 0.5))
            return spans

        final_spans: List[Optional[Tuple[int, int, float]]] = [None] * N
        for idx_a, s_f, e_f, conf in anchors:
            final_spans[idx_a] = (s_f, e_f, conf)

        first_idx, first_s, _, _ = anchors[0]
        if first_idx > 0:
            step = max(1, first_s // first_idx)
            for k in range(first_idx):
                s = max(0, min(T - 1, int(k * step)))
                e = max(s, min(T - 1, min(first_s, int((k + 1) * step))))
                final_spans[k] = (s, e, 0.0)

        for a_idx in range(len(anchors) - 1):
            i_curr, _, e_curr, _ = anchors[a_idx]
            i_next, s_next, _, _ = anchors[a_idx + 1]
            gap_count = i_next - i_curr - 1
            if gap_count > 0:
                gap_start = e_curr
                gap_end = max(s_next, gap_start + gap_count)
                step = max(gap_count, gap_end - gap_start) / gap_count
                for step_i in range(gap_count):
                    target_i = i_curr + 1 + step_i
                    s = max(0, min(T - 1, int(gap_start + step_i * step)))
                    e = max(s, min(T - 1, int(gap_start + (step_i + 1) * step)))
                    final_spans[target_i] = (s, e, 0.0)

        last_idx, _, last_e, _ = anchors[-1]
        if last_idx < N - 1:
            rem_count = N - 1 - last_idx
            step = max(rem_count, T - last_e) / rem_count
            for step_i in range(rem_count):
                target_i = last_idx + 1 + step_i
                s = max(0, min(T - 1, int(last_e + step_i * step)))
                e = max(s, min(T - 1, int(last_e + (step_i + 1) * step)))
                final_spans[target_i] = (s, e, 0.0)

        for i in range(N):
            s_f, e_f, conf = final_spans[i]
            spans.append(PhonemeAlignment(phonemes[i], target_list[i], s_f, e_f, conf))
        return spans

    def _pool_node_features(
        self,
        frame_features: torch.Tensor,
        spans: List[PhonemeAlignment],
    ) -> torch.Tensor:
        nodes = []
        T = frame_features.shape[0]
        for span in spans:
            s = max(0, span.start_frame)
            e = min(T - 1, span.end_frame)
            if s <= e:
                pooled = frame_features[s : e + 1].mean(dim=0)
            else:
                pooled = frame_features.mean(dim=0)
            nodes.append(pooled)
        return torch.stack(nodes, dim=0) if nodes else frame_features.new_zeros(0, frame_features.shape[-1])

    def _uniform_align(
        self,
        frame_features: torch.Tensor,
        phonemes: List[str],
    ) -> Tuple[List[PhonemeAlignment], torch.Tensor]:
        T, D = frame_features.shape
        n = max(len(phonemes), 1)
        chunk = T // n
        tids = self.phonemes_to_ids(phonemes).tolist()
        spans = []
        for i, ph in enumerate(phonemes):
            s = i * chunk
            e = min(T - 1, (i + 1) * chunk - 1) if i < n - 1 else max(0, T - 1)
            spans.append(PhonemeAlignment(ph, tids[i], s, e, 0.5))
        node_features = self._pool_node_features(frame_features, spans)
        return spans, node_features

    def batch_align(
        self,
        frame_features: torch.Tensor,
        phoneme_lists: List[List[str]],
        frame_lengths: torch.Tensor,
        batch_log_probs: Optional[torch.Tensor] = None,
    ) -> List[Tuple[List[PhonemeAlignment], torch.Tensor]]:
        results = []
        B = frame_features.shape[0]
        for b in range(B):
            T_b = int(frame_lengths[b].item())
            lp_b = batch_log_probs[b : b + 1, :T_b] if batch_log_probs is not None else None
            spans, nodes = self.align_utterance(
                frame_features[b, :T_b], phoneme_lists[b], T_b, log_probs=lp_b
            )
            results.append((spans, nodes))
        return results

    def ctc_loss(
        self,
        frame_features: torch.Tensor,
        phoneme_lists: List[List[str]],
        frame_lengths: torch.Tensor,
        batch_log_probs: Optional[torch.Tensor] = None,
    ) -> torch.Tensor:
        if batch_log_probs is not None:
            log_probs = batch_log_probs.float().transpose(0, 1)
        else:
            log_probs = self.forward_ctc_logits(frame_features)
        targets_list = []
        target_lengths = []
        for phs in phoneme_lists:
            t = self.phonemes_to_ids(phs)
            targets_list.append(t)
            target_lengths.append(len(t))
        if not targets_list or sum(target_lengths) == 0:
            return frame_features.new_zeros(())
        targets = torch.cat(targets_list).to(frame_features.device)
        target_lengths_t = torch.tensor(target_lengths, device=frame_features.device)
        return F.ctc_loss(
            log_probs,
            targets,
            frame_lengths,
            target_lengths_t,
            blank=self.blank_id,
            zero_infinity=True,
        )


class PhonemeGraphNetwork(nn.Module):
    def __init__(
        self,
        input_dim: int,
        hidden_dim: int = 256,
        num_layers: int = 2,
        num_heads: int = 4,
        dropout: float = 0.1,
        edge_sequential: bool = True,
        edge_same_word: bool = True,
        edge_same_syllable: bool = False,
    ):
        super().__init__()
        if GATv2Conv is None:
            raise ImportError("torch_geometric is required for PhonemeGraphNetwork")

        self.input_proj = nn.Linear(input_dim, hidden_dim)
        self.edge_sequential = edge_sequential
        self.edge_same_word = edge_same_word
        self.edge_same_syllable = edge_same_syllable

        self.gat_layers = nn.ModuleList()
        for _ in range(num_layers):
            self.gat_layers.append(
                GATv2Conv(
                    in_channels=hidden_dim,
                    out_channels=hidden_dim // num_heads,
                    heads=num_heads,
                    dropout=dropout,
                    concat=True,
                )
            )
        self.norm = nn.LayerNorm(hidden_dim)
        self.dropout = nn.Dropout(dropout)
        self.output_dim = hidden_dim

    @staticmethod
    def build_edge_index(
        num_phonemes: int,
        word_phone_ranges: List[Tuple[int, int]],
        sequential: bool = True,
        same_word: bool = True,
        same_syllable: bool = False,
    ) -> torch.Tensor:
        edges = set()
        if sequential:
            for i in range(num_phonemes - 1):
                edges.add((i, i + 1))
                edges.add((i + 1, i))
        if same_word:
            for start, end in word_phone_ranges:
                s = max(0, start)
                e = min(num_phonemes, end)
                for i in range(s, e):
                    for j in range(s, e):
                        if i != j:
                            edges.add((i, j))
        if same_syllable:
            for start, end in word_phone_ranges:
                s = max(0, start)
                e = min(num_phonemes, end)
                mid = (s + e) // 2
                for i in range(s, mid):
                    for j in range(s, mid):
                        if i != j:
                            edges.add((i, j))
                for i in range(mid, e):
                    for j in range(mid, e):
                        if i != j:
                            edges.add((i, j))
        if not edges:
            edges.add((0, 0))
        src, dst = zip(*edges)
        return torch.tensor([src, dst], dtype=torch.long)

    def forward_single(
        self,
        node_features: torch.Tensor,
        word_phone_ranges: List[Tuple[int, int]],
    ) -> torch.Tensor:
        x = self.input_proj(node_features)
        edge_index = self.build_edge_index(
            node_features.shape[0],
            word_phone_ranges,
            self.edge_sequential,
            self.edge_same_word,
            self.edge_same_syllable,
        ).to(node_features.device)

        for gat in self.gat_layers:
            x = gat(x, edge_index)
            x = self.dropout(torch.relu(x))
        return self.norm(x)

    def forward_batch(
        self,
        node_features_list: List[torch.Tensor],
        word_phone_ranges_list: List[List[Tuple[int, int]]],
    ) -> List[torch.Tensor]:
        outputs = []
        for nodes, ranges in zip(node_features_list, word_phone_ranges_list):
            if nodes.shape[0] == 0:
                outputs.append(nodes.new_zeros(0, self.output_dim))
            else:
                outputs.append(self.forward_single(nodes, ranges))
        return outputs


class PhonemeMDDHead(nn.Module):
    def __init__(self, input_dim: int, num_classes: int = 4, dropout: float = 0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.LayerNorm(input_dim),
            nn.Dropout(dropout),
            nn.Linear(input_dim, num_classes),
        )

    def forward(self, node_features: torch.Tensor) -> torch.Tensor:
        return self.net(node_features)


class MDDLoss(nn.Module):
    """Weighted Cross-Entropy + Focal Loss with global confusion stats."""

    def __init__(self, class_weights: Optional[List[float]] = None, gamma: float = 1.5):
        super().__init__()
        w = torch.tensor(class_weights or [1.0, 3.0, 6.0, 8.0], dtype=torch.float32)
        self.register_buffer("weight", w)
        self.gamma = float(gamma)

    def forward(
        self,
        logits_list: List[torch.Tensor],
        label_lists: List[List[int]],
    ) -> Dict[str, Any]:
        device = logits_list[0].device if logits_list else self.weight.device
        weight = self.weight.to(device)
        losses, all_pred, all_true = [], [], []
        for logits, labels in zip(logits_list, label_lists):
            n = min(logits.shape[0], len(labels))
            if n == 0:
                continue
            tgt = torch.tensor(labels[:n], device=device, dtype=torch.long)
            logits_f = logits[:n].float()
            ce = F.cross_entropy(logits_f, tgt, reduction="none")
            if self.gamma > 0:
                pt = torch.exp(-ce)
                focal = (1.0 - pt) ** self.gamma
            else:
                focal = 1.0
            alpha_t = weight[tgt]
            losses.append((alpha_t * focal * ce).mean())
            all_pred.append(logits_f.argmax(-1))
            all_true.append(tgt)

        if not losses:
            z = torch.tensor(0.0, device=device, requires_grad=True)
            return {"loss": z, "accuracy": z, "tp": 0, "fp": 0, "fn": 0, "tn": 0}

        loss = torch.stack(losses).mean()
        pred = torch.cat(all_pred)
        true = torch.cat(all_true)
        acc = (pred == true).float().mean()
        err_pred = pred != ERR_CORRECT
        err_true = true != ERR_CORRECT
        tp = int((err_pred & err_true).sum().item())
        fp = int((err_pred & ~err_true).sum().item())
        fn = int((~err_pred & err_true).sum().item())
        tn = int((~err_pred & ~err_true).sum().item())
        return {"loss": loss, "accuracy": acc, "tp": tp, "fp": fp, "fn": fn, "tn": tn}


class L2MDDModel(nn.Module):
    def __init__(self, config: Dict[str, Any]):
        super().__init__()
        wavlm_cfg = config.get("wavlm", {})
        trans_cfg = config.get("transformer", {})
        graph_cfg = config.get("phoneme_graph", {})
        mdd_cfg = config.get("mdd", {})

        self.wavlm = WavLMEncoder(
            model_name=wavlm_cfg.get("model_name", "microsoft/wavlm-large"),
            freeze=wavlm_cfg.get("freeze", True),
            use_lora=wavlm_cfg.get("use_lora", False),
            lora_r=wavlm_cfg.get("lora_r", 8),
            lora_alpha=wavlm_cfg.get("lora_alpha", 16),
            lora_dropout=wavlm_cfg.get("lora_dropout", 0.05),
            lora_target_modules=wavlm_cfg.get("lora_target_modules"),
        )
        d = self.wavlm.output_dim

        self.task_transformer = TaskTransformerEncoder(
            input_dim=d,
            num_layers=trans_cfg.get("num_layers", 3),
            num_heads=trans_cfg.get("num_heads", 8),
            ff_dim=trans_cfg.get("ff_dim", 2048),
            dropout=trans_cfg.get("dropout", 0.1),
            max_seq_len=trans_cfg.get("max_seq_len", 800),
        )

        self.ctc_aligner = CTCAligner(input_dim=d, use_pretrained_bundle=False)
        canon_emb_dim = int(mdd_cfg.get("canon_emb_dim", 256))
        self.use_canon_conditioning = True
        self.canon_emb = nn.Embedding(self.ctc_aligner.num_tokens, canon_emb_dim)
        self.node_fusion = nn.Sequential(
            nn.Linear(d + canon_emb_dim + 4, d),
            nn.LayerNorm(d),
            nn.GELU(),
        )

        graph_hidden = graph_cfg.get("hidden_dim", 256)
        self.phoneme_graph = PhonemeGraphNetwork(
            input_dim=d,
            hidden_dim=graph_hidden,
            num_layers=graph_cfg.get("num_gat_layers", 2),
            num_heads=graph_cfg.get("num_heads", 4),
            dropout=graph_cfg.get("dropout", 0.1),
            edge_sequential=graph_cfg.get("edge_types", {}).get("sequential", True),
            edge_same_word=graph_cfg.get("edge_types", {}).get("same_word", True),
            edge_same_syllable=graph_cfg.get("edge_types", {}).get("same_syllable", False),
        )
        self.mdd_head = PhonemeMDDHead(
            graph_hidden,
            num_classes=mdd_cfg.get("num_classes", 4),
            dropout=mdd_cfg.get("dropout", 0.1),
        )

    def _condition_nodes(
        self,
        acoustic_nodes: torch.Tensor,
        ctc_log_probs_uttr: torch.Tensor,
        spans: List[PhonemeAlignment],
        phoneme_tokens: List[str],
    ) -> torch.Tensor:
        if acoustic_nodes.shape[0] == 0:
            return acoustic_nodes
        N = acoustic_nodes.shape[0]
        device = acoustic_nodes.device
        target_ids = self.ctc_aligner.phonemes_to_ids(phoneme_tokens[:N]).to(device)
        canon_vecs = self.canon_emb(target_ids)

        T_u = ctc_log_probs_uttr.shape[0]
        probs = ctc_log_probs_uttr.detach().float().exp()
        cues = []
        for i in range(N):
            tid = int(target_ids[i].item())
            sp = spans[i] if i < len(spans) else None
            if sp is not None and T_u > 0:
                sf = max(0, min(T_u - 1, int(sp.start_frame)))
                ef = max(sf, min(T_u - 1, int(sp.end_frame)))
                seg = probs[sf : ef + 1]
                p_tgt = seg[:, tid].max() if tid < seg.shape[1] else acoustic_nodes.new_tensor(0.0)
                p_blk = seg[:, 0].mean()
                other = seg.clone()
                other[:, :3] = 0.0
                if tid < other.shape[1]:
                    other[:, tid] = 0.0
                p_oth = other.max()
                dur_norm = min(1.0, (ef - sf + 1) / 10.0)
                cues.append(torch.stack([p_tgt, p_blk, p_oth, p_tgt.new_tensor(dur_norm)]))
            else:
                cues.append(acoustic_nodes.new_zeros(4))
        cues_t = torch.stack(cues, dim=0).to(device=device, dtype=acoustic_nodes.dtype)
        fused = torch.cat([acoustic_nodes, canon_vecs.to(dtype=acoustic_nodes.dtype), cues_t], dim=-1)
        return self.node_fusion(fused)

    def forward(
        self,
        waveforms: torch.Tensor,
        wav_lengths: torch.Tensor,
        phoneme_tokens: List[List[str]],
        word_phone_ranges: List[List[tuple]],
    ) -> Dict[str, Any]:
        frame_feats = self.wavlm(waveforms, wav_lengths=wav_lengths)
        frame_lengths = self.wavlm.frame_lengths_from_samples(wav_lengths)

        T = frame_feats.shape[1]
        pad_mask = torch.arange(T, device=waveforms.device).unsqueeze(0) >= frame_lengths.unsqueeze(1)
        frame_feats = self.task_transformer(frame_feats, src_key_padding_mask=pad_mask)

        ctc_logits = self.ctc_aligner.ctc_proj(frame_feats).float()
        batch_log_probs = F.log_softmax(ctc_logits, dim=-1)

        align_results = self.ctc_aligner.batch_align(
            frame_feats, phoneme_tokens, frame_lengths, batch_log_probs=batch_log_probs
        )
        node_features_list = []
        for b, (spans, acoustic_nodes) in enumerate(align_results):
            T_b = int(frame_lengths[b].item())
            cond_nodes = self._condition_nodes(
                acoustic_nodes, batch_log_probs[b, :T_b], spans, phoneme_tokens[b]
            )
            node_features_list.append(cond_nodes)

        graph_out = self.phoneme_graph.forward_batch(node_features_list, word_phone_ranges)
        logits_list = [self.mdd_head(g) for g in graph_out]

        ctc_loss = self.ctc_aligner.ctc_loss(
            frame_feats, phoneme_tokens, frame_lengths, batch_log_probs=batch_log_probs
        )

        return {
            "mdd_logits": logits_list,
            "ctc_loss": ctc_loss,
            "graph_embeddings": graph_out,
        }

    @classmethod
    def from_checkpoint(cls, path: str, config: Dict[str, Any], device: str = "cpu") -> "L2MDDModel":
        model = cls(config).to(device)
        st = torch.load(path, map_location=device, weights_only=False)
        model.load_state_dict(st["model_state_dict"] if isinstance(st, dict) and "model_state_dict" in st else st)
        model.eval()
        return model


logger = logging.getLogger(__name__)


def inspect_dataset_diagnostics(ds: Dataset, aligner: CTCAligner, name: str) -> List[int]:
    """Print class distribution and <unk> mapping rate before training starts."""
    counts = [0, 0, 0, 0]
    total_ph = 0
    unk_ph = 0
    unmapped_examples = set()
    unk_id = aligner.token2id["<unk>"]
    for s in ds.samples:
        ids = aligner.phonemes_to_ids(s["phoneme_tokens"]).tolist()
        for tok, tid, lbl in zip(s["phoneme_tokens"], ids, s["phoneme_labels"]):
            total_ph += 1
            if 0 <= lbl < 4:
                counts[lbl] += 1
            if tid == unk_id:
                unk_ph += 1
                if len(unmapped_examples) < 10:
                    unmapped_examples.add(tok)
    unk_pct = 100.0 * unk_ph / max(1, total_ph)
    dist = " | ".join(
        f"{ERR_NAMES[c]}={counts[c]} ({100.0 * counts[c] / max(1, total_ph):.1f}%)"
        for c in range(4)
    )
    print(f"[{name}] Phonemes={total_ph:,} | <unk>={unk_ph} ({unk_pct:.2f}%) | {dist}")
    if unmapped_examples:
        print(f"   Unmapped sample tokens: {sorted(unmapped_examples)}")
    assert unk_pct < 5.0, f"Too many <unk> phonemes ({unk_pct:.2f}%): check TextGrid normalization!"
    return counts


class MDDTrainer:
    def __init__(self, config: Dict[str, Any]):
        self.config = config
        tc = config["train"]
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.use_amp = bool(tc.get("use_amp", False)) and self.device.type == "cuda"
        self.scaler = torch.amp.GradScaler("cuda") if self.use_amp else None
        ds_kw = dict(
            sample_rate=tc["dataset"]["sample_rate"],
            max_duration_sec=tc["dataset"]["max_duration_sec"],
            data_dir=tc["dataset"].get("data_dir"),
        )
        self.train_ds = L2ArcticMDDDataset("train", **ds_kw)
        self.val_ds = L2ArcticMDDDataset("test", **ds_kw)
        if len(self.train_ds) == 0:
            raise FileNotFoundError("No L2-ARCTIC annotated samples found.")
        pin = bool(tc.get("pin_memory", False)) and self.device.type == "cuda"
        self.train_loader = DataLoader(
            self.train_ds, tc["batch_size"], shuffle=True,
            num_workers=tc.get("num_workers", 2), collate_fn=collate_mdd_fn, pin_memory=pin,
        )
        self.val_loader = DataLoader(
            self.val_ds, tc["batch_size"], shuffle=False,
            num_workers=tc.get("num_workers", 2), collate_fn=collate_mdd_fn, pin_memory=pin,
        )
        self.model = L2MDDModel(config).to(self.device)

        train_counts = inspect_dataset_diagnostics(self.train_ds, self.model.ctc_aligner, "Train")
        inspect_dataset_diagnostics(self.val_ds, self.model.ctc_aligner, "Test/Val")

        mdd_cfg = config.get("mdd", {})
        cw = mdd_cfg.get("class_weights")
        if cw is None:
            n0 = max(1, train_counts[0])
            cw = [1.0]
            for c in range(1, 4):
                nc = max(1, train_counts[c])
                cw.append(round(min(8.0, (n0 / nc) ** 0.55), 2))
            print(f"[Loss] Auto-computed balanced class_weights = {cw} (focal_gamma={mdd_cfg.get('focal_gamma', 1.5)})")

        self.criterion = MDDLoss(cw, gamma=mdd_cfg.get("focal_gamma", 1.5)).to(self.device)
        self.ctc_weight = tc.get("ctc_weight", 0.5)

        wavlm_params = [p for n, p in self.model.named_parameters() if "wavlm" in n and p.requires_grad]
        head_params = [p for n, p in self.model.named_parameters() if "wavlm" not in n and p.requires_grad]
        lr_wavlm = tc.get("learning_rate_wavlm", 4.0e-5)
        lr_heads = tc.get("learning_rate", 1.5e-4)
        self.optimizer = torch.optim.AdamW([
            {"params": wavlm_params, "lr": lr_wavlm},
            {"params": head_params, "lr": lr_heads},
        ], weight_decay=tc.get("weight_decay", 0.02))

        self.num_epochs = tc["num_epochs"]
        warmup_epochs = tc.get("warmup_epochs", 2)
        warmup_sched = torch.optim.lr_scheduler.LinearLR(
            self.optimizer, start_factor=0.2, end_factor=1.0, total_iters=warmup_epochs
        )
        cosine_sched = torch.optim.lr_scheduler.CosineAnnealingLR(
            self.optimizer, T_max=max(1, self.num_epochs - warmup_epochs), eta_min=1e-6
        )
        self.scheduler = torch.optim.lr_scheduler.SequentialLR(
            self.optimizer, schedulers=[warmup_sched, cosine_sched], milestones=[warmup_epochs]
        )

        self.grad_clip = tc.get("grad_clip", 1.0)
        self.grad_accum = max(1, int(tc.get("grad_accum_steps", 1)))
        self.save_every = tc.get("save_every_epochs", 5)
        self.early_stopping_patience = tc.get("early_stopping_patience", 6)
        self.ckpt_dir = Path(config["paths"]["checkpoint_dir"])
        self.ckpt_dir.mkdir(parents=True, exist_ok=True)

    @staticmethod
    def _global_metrics(tp: int, fp: int, fn: int, tn: int) -> Tuple[float, float, float, float]:
        prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        rec = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f1 = (2 * prec * rec / (prec + rec)) if (prec + rec) > 0 else 0.0
        far = fp / (fp + tn) if (fp + tn) > 0 else 0.0
        return prec, rec, f1, far

    def _step(self, batch, train=True, accum_scale=1.0):
        waveforms = batch["waveforms"].to(self.device)
        wav_lengths = batch["wav_lengths"].to(self.device)
        ctx = (lambda: torch.amp.autocast("cuda")) if (train and self.use_amp) else nullcontext
        with ctx():
            out = self.model(waveforms, wav_lengths, batch["phoneme_tokens"], batch["word_phone_ranges"])
            ld = self.criterion(out["mdd_logits"], batch["phoneme_labels"])
            mdd_l = ld["loss"]
            ctc_l = out["ctc_loss"]
            loss = (mdd_l + self.ctc_weight * ctc_l) / accum_scale
        if train:
            if self.use_amp:
                self.scaler.scale(loss).backward()
            else:
                loss.backward()
        return {
            "loss": float(loss.detach().cpu()) * accum_scale,
            "mdd_loss": float(mdd_l.detach().cpu()),
            "ctc_loss": float(ctc_l.detach().cpu()),
            "tp": ld["tp"], "fp": ld["fp"], "fn": ld["fn"], "tn": ld["tn"],
        }

    def _optimizer_step(self):
        if self.use_amp:
            if self.grad_clip:
                self.scaler.unscale_(self.optimizer)
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), self.grad_clip)
            self.scaler.step(self.optimizer)
            self.scaler.update()
        else:
            if self.grad_clip:
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), self.grad_clip)
            self.optimizer.step()
        self.optimizer.zero_grad()

    def train(self):
        logging.basicConfig(level=logging.INFO)
        logger.info(
            "Device=%s train=%d val=%d batch_size=%d grad_accum=%d ctc_weight=%.2f",
            self.device, len(self.train_ds), len(self.val_ds),
            self.config["train"]["batch_size"], self.grad_accum, self.ctc_weight,
        )
        best_val_loss = float("inf")
        best_f1 = -1.0
        patience_counter = 0

        for epoch in range(1, self.num_epochs + 1):
            t0 = time.time()
            self.model.train()
            tr_sum = {"loss": 0.0, "mdd_loss": 0.0, "ctc_loss": 0.0}
            tr_cm = [0, 0, 0, 0]
            n = 0
            self.optimizer.zero_grad()
            for i, batch in enumerate(tqdm(self.train_loader, desc=f"Epoch {epoch}")):
                ld = self._step(batch, True, self.grad_accum)
                for k in tr_sum:
                    tr_sum[k] += ld[k]
                tr_cm[0] += ld["tp"]; tr_cm[1] += ld["fp"]; tr_cm[2] += ld["fn"]; tr_cm[3] += ld["tn"]
                n += 1
                if i % self.grad_accum == self.grad_accum - 1 or i == len(self.train_loader) - 1:
                    self._optimizer_step()

            self.model.eval()
            va_sum = {"loss": 0.0, "mdd_loss": 0.0, "ctc_loss": 0.0}
            va_cm = [0, 0, 0, 0]
            m = 0
            with torch.no_grad():
                for batch in tqdm(self.val_loader, desc="Val"):
                    ld = self._step(batch, False)
                    for k in va_sum:
                        va_sum[k] += ld[k]
                    va_cm[0] += ld["tp"]; va_cm[1] += ld["fp"]; va_cm[2] += ld["fn"]; va_cm[3] += ld["tn"]
                    m += 1

            if self.device.type == "cuda":
                torch.cuda.empty_cache()

            tr = {k: v / max(n, 1) for k, v in tr_sum.items()}
            va = {k: v / max(m, 1) for k, v in va_sum.items()}
            _, _, tr_f1, _ = self._global_metrics(*tr_cm)
            va_prec, va_rec, va_f1, va_far = self._global_metrics(*va_cm)

            self.scheduler.step()
            current_lrs = [f"{g['lr']:.2e}" for g in self.optimizer.param_groups]
            logger.info(
                "Epoch %d (%.0fs, lr=%s) train loss=%.4f (mdd=%.3f ctc=%.3f) f1=%.3f | "
                "val loss=%.4f (ctc=%.3f) f1=%.3f P=%.3f R=%.3f FAR=%.3f",
                epoch, time.time() - t0, "/".join(current_lrs),
                tr["loss"], tr["mdd_loss"], tr["ctc_loss"], tr_f1,
                va["loss"], va["ctc_loss"], va_f1, va_prec, va_rec, va_far,
            )

            val_loss = va["loss"]
            is_improved = False
            if best_f1 >= 0.05:
                # Strictly select best model by global error-detection F1 once non-trivial
                if va_f1 > best_f1:
                    best_f1 = va_f1
                    best_val_loss = val_loss
                    is_improved = True
            else:
                if va_f1 >= 0.05 or (va_f1 > best_f1 and va_f1 > 0.01):
                    best_f1 = va_f1
                    best_val_loss = val_loss
                    is_improved = True
                elif val_loss < best_val_loss:
                    best_val_loss = val_loss
                    is_improved = True

            if is_improved or epoch == 1:
                if is_improved:
                    patience_counter = 0
                torch.save(self.model.state_dict(), self.ckpt_dir / "best_model.pt")
                logger.info(
                    "   🌟 Best model updated (val_f1=%.3f, val_loss=%.4f) -> checkpoints/l2_mdd/best_model.pt",
                    va_f1, val_loss,
                )
            else:
                patience_counter += 1
                logger.info(
                    "   (Validation did not improve. Patience: %d/%d)",
                    patience_counter, self.early_stopping_patience,
                )
                if patience_counter >= self.early_stopping_patience:
                    logger.info("   ⏹️ Early stopping triggered at epoch %d", epoch)
                    break

            if self.save_every and epoch % self.save_every == 0:
                torch.save(
                    {"epoch": epoch, "model_state_dict": self.model.state_dict(), "config": self.config},
                    self.ckpt_dir / f"epoch_{epoch}.pt",
                )

        best_ckpt = self.ckpt_dir / "best_model.pt"
        if not best_ckpt.is_file():
            torch.save(self.model.state_dict(), best_ckpt)
        self.model.load_state_dict(torch.load(best_ckpt, map_location=self.device, weights_only=False))
        logger.info("✅ Best model weights restored successfully! (Best val_f1=%.3f)", max(0.0, best_f1))

    def evaluate_test_set(self):
        """Đánh giá chi tiết trên tập kiểm thử L2-ARCTIC và in báo cáo phân loại, ma trận nhầm lẫn."""
        import numpy as np

        ckpt_path = self.ckpt_dir / "best_model.pt"
        if ckpt_path.is_file():
            self.model.load_state_dict(torch.load(ckpt_path, map_location=self.device, weights_only=False))
        self.model.eval()

        all_preds = []
        all_trues = []

        print(f"⚡ Đang chạy đánh giá trên tập test ({len(self.val_ds)} bài kiểm thử L2-ARCTIC)...")
        with torch.no_grad():
            for batch in tqdm(self.val_loader, desc="Testing L2-MDD"):
                waveforms = batch["waveforms"].to(self.device)
                wav_lengths = batch["wav_lengths"].to(self.device)
                out = self.model(waveforms, wav_lengths, batch["phoneme_tokens"], batch["word_phone_ranges"])
                logits_list = out["mdd_logits"]
                label_lists = batch["phoneme_labels"]

                for logits, labels in zip(logits_list, label_lists):
                    n = min(logits.shape[0], len(labels))
                    if n > 0:
                        all_preds.extend(logits[:n].argmax(dim=-1).cpu().tolist())
                        all_trues.extend(labels[:n])

        if not all_trues:
            print("❌ Không có dữ liệu kiểm thử L2-MDD.")
            return

        y_true = np.array(all_trues)
        y_pred = np.array(all_preds)
        total_phonemes = len(y_true)

        class_names = ["Correct (Chuẩn)", "Substitution (Thay thế)", "Deletion (Bỏ âm)", "Addition (Thêm âm)"]
        short_names = ["Correct", "Sub", "Del", "Add"]

        cm = np.zeros((4, 4), dtype=int)
        for t, p in zip(y_true, y_pred):
            if 0 <= t < 4 and 0 <= p < 4:
                cm[t, p] += 1

        overall_acc = float(np.mean(y_true == y_pred) * 100)

        err_true = (y_true != 0)
        err_pred = (y_pred != 0)
        tp = int(np.sum(err_true & err_pred))
        fp = int(np.sum(~err_true & err_pred))
        fn = int(np.sum(err_true & ~err_pred))
        tn = int(np.sum(~err_true & ~err_pred))

        det_prec = (tp / (tp + fp) * 100) if (tp + fp) > 0 else 0.0
        det_rec = (tp / (tp + fn) * 100) if (tp + fn) > 0 else 0.0
        det_f1 = (2 * det_prec * det_rec / (det_prec + det_rec)) if (det_prec + det_rec) > 0 else 0.0
        far = (fp / (fp + tn) * 100) if (fp + tn) > 0 else 0.0

        print("\n" + "=" * 80)
        print("📊 BÁO CÁO ĐÁNH GIÁ HIỆU NĂNG MÔ HÌNH L2-MDD TRÊN TẬP TEST L2-ARCTIC")
        print("=" * 80)
        print(f"Tổng số âm vị đánh giá (Phonemes): {total_phonemes:,}")
        print(f" • Độ chính xác tổng thể (Phoneme Accuracy): {overall_acc:.2f}%")
        print(f" • Chỉ số F1 phát hiện lỗi âm (Error Detection F1): {det_f1:.2f}%")
        print(f" • Độ chuẩn xác phát hiện lỗi (Precision): {det_prec:.2f}%")
        print(f" • Độ nhạy / Độ bao phủ lỗi (Recall): {det_rec:.2f}%")
        print(f" • Tỷ lệ báo động giả (False Alarm Rate - FAR): {far:.2f}%")
        print("=" * 80)

        print("\n🎯 CHI TIẾT HIỆU NĂNG TỪNG LOẠI LỖI PHÁT ÂM:")
        print(f"{'Loại Lỗi':<28} | {'Số Mẫu':<10} | {'Precision (%)':<15} | {'Recall (%)':<12} | {'F1-Score (%)':<12}")
        print("-" * 84)
        for c in range(4):
            cnt = int(np.sum(y_true == c))
            c_tp = cm[c, c]
            c_pred_cnt = int(np.sum(y_pred == c))
            prec_c = (c_tp / c_pred_cnt * 100) if c_pred_cnt > 0 else 0.0
            rec_c = (c_tp / cnt * 100) if cnt > 0 else 0.0
            f1_c = (2 * prec_c * rec_c / (prec_c + rec_c)) if (prec_c + rec_c) > 0 else 0.0
            print(f"{class_names[c]:<28} | {cnt:<10} | {prec_c:<15.2f} | {rec_c:<12.2f} | {f1_c:<12.2f}")
        print("=" * 84)

        print("\n🧩 MA TRẬN NHẦM LẪN (CONFUSION MATRIX):")
        hdr_title = "True \\ Pred"
        print(f"{hdr_title:<14} | {'Correct':<10} | {'Sub':<10} | {'Del':<10} | {'Add':<10}")
        print("-" * 62)
        for r in range(4):
            row_str = f"{short_names[r]:<14} | " + " | ".join(f"{cm[r, c]:<10}" for c in range(4))
            print(row_str)
        print("=" * 62)

In [ ]:
import os, shutil, torch
os.environ.setdefault("PYTORCH_ALLOC_CONF", "expandable_segments:True")
os.makedirs("/kaggle/working/hf_cache", exist_ok=True)
os.environ["HF_HOME"] = os.environ["TRANSFORMERS_CACHE"] = "/kaggle/working/hf_cache"
assert torch.cuda.is_available(), "Enable GPU T4"
torch.cuda.empty_cache()

data_dir = prepare_l2_arctic_dir()
CONFIG["train"]["dataset"]["data_dir"] = str(data_dir)
ok, issues, info = verify_l2_arctic(data_dir)
print("Dataset:", info)
if not ok:
    raise RuntimeError("\n".join(issues))

train_preview = L2ArcticMDDDataset("train", data_dir=str(data_dir))
test_preview = L2ArcticMDDDataset("test", data_dir=str(data_dir))
print("Samples:", len(train_preview), "train,", len(test_preview), "test")
print("Sample #0 phonemes:", train_preview.samples[0]["phoneme_tokens"][:15])
print("Sample #0 labels:  ", train_preview.samples[0]["phoneme_labels"][:15])

trainer = MDDTrainer(CONFIG)
trainer.train()

from pathlib import Path
ckpt_src = Path("/kaggle/working/checkpoints/l2_mdd/best_model.pt")
if not ckpt_src.is_file():
    ckpt_src.parent.mkdir(parents=True, exist_ok=True)
    torch.save(trainer.model.state_dict(), ckpt_src)
shutil.copy(str(ckpt_src), "/kaggle/working/l2_mdd_best.pt")

print("\n" + "=" * 80)
print("🎯 BẮT ĐẦU ĐÁNH GIÁ CHI TIẾT TRÊN TẬP TEST L2-ARCTIC...")
print("=" * 80)
trainer.evaluate_test_set()
print("\n✅ Hoàn tất! Trọng số tối ưu đã được lưu tại: /kaggle/working/l2_mdd_best.pt")